In [20]:
# ============================================================
# 1. PREPARE RANDOM LABEL-HOLDOUT DATASET
# ============================================================

feature_columns = [
    col
    for col in base_df.columns
    if col not in ["txId", "time_step", "class"]
]

# ------------------------------------------------------------
# Unlabeled transactions
# Used for model training ONLY
# ------------------------------------------------------------

unsupervised_random_df = (
    base_df[
        base_df["class"] == "unknown"
    ]
    .copy()
)

# ------------------------------------------------------------
# Labeled transactions
# Used for evaluation ONLY
# ------------------------------------------------------------

labeled_random_df = (
    base_df[
        base_df["class"] != "unknown"
    ]
    .copy()
)

labeled_random_df["is_illicit"] = (
    labeled_random_df["class"] == "1"
).astype(int)

print("Full dataset:", base_df.shape)

print(
    "Unlabeled training data:",
    unsupervised_random_df.shape
)

print(
    "Labeled evaluation data:",
    labeled_random_df.shape
)

Full dataset: (203769, 168)
Unlabeled training data: (157205, 168)
Labeled evaluation data: (46564, 169)


In [21]:
print("=" * 60)
print("FEATURE MATRICES and LABELS")
print("=" * 60)

X_train_random = (
    unsupervised_random_df[
        feature_columns
    ]
    .copy()
)

X_test_random = (
    labeled_random_df[
        feature_columns
    ]
    .copy()
)

print("\nFeature matrices:")
print("X_train_random:", X_train_random.shape)
print("X_test_random: ", X_test_random.shape)

y_test_random = (
    labeled_random_df["is_illicit"]
    .astype(int)
    .copy()
)

FEATURE MATRICES and LABELS

Feature matrices:
X_train_random: (157205, 165)
X_test_random:  (46564, 165)


In [117]:
print("=" * 60)
print("RANDOM LABEL-HOLDOUT SPLIT")
print("=" * 60)

print(
    "Training transactions:",
    len(unsupervised_random_df)
)

print(
    "Evaluation transactions:",
    len(labeled_random_df)
)

print(
    "Training contains labels:",
    "class" in unsupervised_random_df.columns
)

print(
    "Evaluation illicit:",
    int(y_test_random.sum())
)

print(
    "Evaluation licit:",
    int((y_test_random == 0).sum())
)

RANDOM LABEL-HOLDOUT SPLIT
Training transactions: 157205
Evaluation transactions: 46564
Training contains labels: True
Evaluation illicit: 4545
Evaluation licit: 42019


In [118]:
# ============================================================
# SHARED EVAL HELPER
# (same helper as in 03_training.ipynb -- duplicated here so this
# notebook can run standalone without depending on 03_training's
# kernel state)
# ============================================================

def evaluate_scores(y_true, y_score, label):
    roc = roc_auc_score(y_true, y_score)
    pr = average_precision_score(y_true, y_score)
    print("=" * 60)
    print(label)
    print("=" * 60)
    print(f"ROC-AUC: {roc:.6f}")
    print(f"PR-AUC:  {pr:.6f}")
    return roc, pr


def create_anomaly_predictions(model, X, df, extra_columns=None):
    """
    higher = more anomalous (score_samples is negated).
    """
    data = {
        "txId": df["txId"].values,
        "time_step": df["time_step"].values,
        "anomaly_score": -model.score_samples(X),
    }
    if extra_columns:
        data.update(extra_columns)
    return pd.DataFrame(data)


In [119]:
print("=" * 60)
print("LOF")
print("=" * 60)

selected_lof = create_lof_pipeline(
    n_neighbors=125,
    contamination=0.05,
    metric="euclidean",
    p=2,
    leaf_size=30,
)

selected_lof.fit(X_train_random)

print("LOF trained successfully.")
print("Training shape:", X_train_random.shape)

joblib.dump(selected_lof, "models/lof/lof_randomsplit.joblib")

LOF
LOF trained successfully.
Training shape: (157205, 165)


['models/lof/lof_randomsplit.joblib']

In [120]:
print("=" * 60)
print("EVALUATION")
print("=" * 60)

random_lof_predictions = create_anomaly_predictions(
    selected_lof, X_test_random, labeled_random_df
)

random_lof_eval = random_lof_predictions.assign(
    is_illicit=y_test_random.values
)

random_lof_roc_auc, random_lof_pr_auc = evaluate_scores(
    random_lof_eval["is_illicit"],
    random_lof_eval["anomaly_score"],
    "LOF — RANDOM LABEL-HOLDOUT",
)


EVALUATION
LOF — RANDOM LABEL-HOLDOUT
ROC-AUC: 0.227117
PR-AUC:  0.058514


In [121]:
print("=" * 60)
print("ISOLATION FOREST")
print("=" * 60)

random_if = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)

random_if.fit(X_train_random)

print("Isolation Forest trained successfully.")
print("Training shape:", X_train_random.shape)

ISOLATION FOREST
Isolation Forest trained successfully.
Training shape: (157205, 165)


In [122]:
random_if_predictions = create_anomaly_predictions(
    random_if, X_test_random, labeled_random_df
)

random_if_eval = random_if_predictions.assign(
    is_illicit=y_test_random.values
)

print("Evaluation transactions:", len(random_if_eval))
print("\nClass distribution:")
print(random_if_eval["is_illicit"].value_counts())
display(random_if_eval.head())

random_if_roc_auc, random_if_pr_auc = evaluate_scores(
    random_if_eval["is_illicit"],
    random_if_eval["anomaly_score"],
    "ISOLATION FOREST — RANDOM LABEL-HOLDOUT",
)


Evaluation transactions: 46564

Class distribution:
is_illicit
0    42019
1     4545
Name: count, dtype: int64


,txId,time_step,anomaly_score,is_illicit
0,232438397,1,0.559692,0
1,232029206,1,0.528558,0
2,232344069,1,0.341704,0
3,27553029,1,0.345623,0
4,3881097,1,0.529497,0


ISOLATION FOREST — RANDOM LABEL-HOLDOUT
ROC-AUC: 0.094723
PR-AUC:  0.052174
